# Notebook 01 — Exploratory Data Analysis

**Project**: Diabetes Prediction using the Diabetes Health Indicators Dataset  
**Target**: `Diabetes_binary` (classes 0 = no diabetes, 1 = diabetes)

This notebook covers:
1. Imports and setup
2. Loading the dataset
3. Basic info (shape, dtypes, head)
4. Missing value analysis
5. Target variable distribution
6. Statistical summary
7. Feature distributions
8. Correlation heatmap
9. Box plots for outlier detection
10. Feature-target relationship plots
11. Key insights

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

from src.data_loader import (
    load_dataset, basic_info, print_basic_info,
    validate_dataset, describe_dataset, get_feature_target_split
)
from src.visualization import (
    plot_distribution, plot_correlation_heatmap, plot_box_plots,
    plot_target_distribution, plot_feature_target_relationship
)
from src.config import RAW_DATASET_PATH, FEATURE_COLUMNS, TARGET_COLUMN, PLOTS_DIR

print('Libraries loaded successfully.')
print(f'Project root : {PROJECT_ROOT}')
print(f'Dataset path : {RAW_DATASET_PATH}')

## 2. Load Dataset

In [ ]:
try:
    df = load_dataset(str(RAW_DATASET_PATH))
    print(f'Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
except FileNotFoundError as e:
    print(str(e))
    print()
    print('Creating a synthetic demo dataset to allow notebook execution...')
    rng = np.random.default_rng(42)
    n = 5000
    df = pd.DataFrame({
        'HighBP':              rng.integers(0, 2, n).astype(float),
        'HighChol':            rng.integers(0, 2, n).astype(float),
        'CholCheck':           rng.integers(0, 2, n).astype(float),
        'BMI':                 rng.normal(28, 6, n).clip(12, 80),
        'Smoker':              rng.integers(0, 2, n).astype(float),
        'Stroke':              rng.integers(0, 2, n).astype(float),
        'HeartDiseaseorAttack':rng.integers(0, 2, n).astype(float),
        'PhysActivity':        rng.integers(0, 2, n).astype(float),
        'Fruits':              rng.integers(0, 2, n).astype(float),
        'Veggies':             rng.integers(0, 2, n).astype(float),
        'HvyAlcoholConsump':   rng.integers(0, 2, n).astype(float),
        'AnyHealthcare':       rng.integers(0, 2, n).astype(float),
        'NoDocbcCost':         rng.integers(0, 2, n).astype(float),
        'GenHlth':             rng.integers(1, 6, n).astype(float),
        'MentHlth':            rng.integers(0, 31, n).astype(float),
        'PhysHlth':            rng.integers(0, 31, n).astype(float),
        'DiffWalk':            rng.integers(0, 2, n).astype(float),
        'Sex':                 rng.integers(0, 2, n).astype(float),
        'Age':                 rng.integers(1, 14, n).astype(float),
        'Education':           rng.integers(1, 7, n).astype(float),
        'Income':              rng.integers(1, 9, n).astype(float),
        'Diabetes_binary':     rng.choice([0, 1], n, p=[0.86, 0.14]).astype(float),
    })
    print(f'Synthetic dataset created: {df.shape}')

## 3. Basic Info

In [ ]:
print('=== BASIC INFO ===')
print_basic_info(df)
print()
print('=== HEAD ===')
df.head()

In [ ]:
print('=== DATA TYPES ===')
print(df.dtypes)
print()
print('=== DATASET VALIDATION ===')
validation = validate_dataset(df)
for key, val in validation.items():
    print(f'  {key}: {val}')

## 4. Missing Value Analysis

In [ ]:
info = basic_info(df)
missing_vals = info['missing_values']

if missing_vals:
    missing_series = pd.Series(missing_vals).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(missing_series.index, missing_series.values, color='coral')
    ax.set_title('Missing Values per Column', fontsize=13, fontweight='bold')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig(str(PLOTS_DIR / 'missing_values.png'), bbox_inches='tight')
    plt.show()
else:
    print('No missing values found in the dataset.')

## 5. Target Variable Distribution

In [ ]:
y = df[TARGET_COLUMN]
print('Class distribution:')
print(y.value_counts().sort_index())
print()
print('Class proportions:')
print(y.value_counts(normalize=True).sort_index().round(4))

plot_target_distribution(y, save_path=str(PLOTS_DIR / 'target_distribution.png'))

## 6. Statistical Summary

In [ ]:
stats = describe_dataset(df)
print('Extended statistical summary (including skewness and kurtosis):')
stats

In [ ]:
# Highlight highly skewed features
high_skew = stats[stats['skewness'].abs() > 1.0]['skewness'].sort_values(ascending=False)
print('Columns with |skewness| > 1:')
print(high_skew)

## 7. Feature Distributions

In [ ]:
continuous_features = ['BMI', 'MentHlth', 'PhysHlth', 'GenHlth', 'Age', 'Education', 'Income']
plot_distribution(
    df, continuous_features,
    save_path=str(PLOTS_DIR / 'feature_distributions.png')
)

In [ ]:
# Binary features
binary_features = [
    'HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke',
    'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
    'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'DiffWalk', 'Sex'
]
print('Binary feature value counts:')
for col in binary_features:
    if col in df.columns:
        counts = df[col].value_counts().sort_index()
        pct = (counts / len(df) * 100).round(1)
        print(f'  {col:<25} {dict(zip(counts.index, pct.values))} %')

## 8. Correlation Heatmap

In [ ]:
plot_correlation_heatmap(df, save_path=str(PLOTS_DIR / 'correlation_heatmap.png'))

In [ ]:
# Top correlations with target
corr_with_target = df.corr()[TARGET_COLUMN].drop(TARGET_COLUMN).sort_values(ascending=False)
print('Top 10 features most correlated with target:')
print(corr_with_target.head(10).round(4))
print()
print('Bottom 5 (most negatively correlated):')
print(corr_with_target.tail(5).round(4))

## 9. Box Plots for Outlier Detection

In [ ]:
numeric_features = ['BMI', 'MentHlth', 'PhysHlth', 'GenHlth', 'Age', 'Education', 'Income']
plot_box_plots(
    df, numeric_features,
    save_path=str(PLOTS_DIR / 'box_plots.png')
)

In [ ]:
# Quantify outliers per column using IQR
from src.preprocessor import detect_outliers_iqr

outlier_mask = detect_outliers_iqr(df, numeric_features)
print('Outlier counts per feature (IQR method, threshold=1.5):')
outlier_counts = outlier_mask.sum().sort_values(ascending=False)
print(outlier_counts)
print(f'\nTotal rows with at least one outlier: {outlier_mask.any(axis=1).sum():,}')

## 10. Feature-Target Relationship Plots

In [ ]:
important_features = [
    'HighBP', 'HighChol', 'BMI', 'GenHlth', 'Age',
    'DiffWalk', 'HeartDiseaseorAttack', 'PhysActivity'
]
plot_feature_target_relationship(
    df, important_features, target_col=TARGET_COLUMN,
    save_path=str(PLOTS_DIR / 'feature_target_relationships.png')
)

In [ ]:
# Cross-tabulations for key binary features
for feat in ['HighBP', 'HighChol', 'GenHlth']:
    if feat in df.columns:
        print(f'\n{feat} vs {TARGET_COLUMN} (normalised by row):')
        ct = pd.crosstab(df[feat], df[TARGET_COLUMN], normalize='index').round(3)
        print(ct)

## 11. Key Insights

### Dataset Summary
- **Size**: ~254k records (or synthetic demo data), 21 features + 1 target column.
- **Target**: `Diabetes_binary` — 2 classes: 0 (no diabetes), 1 (diabetes).
- **Class imbalance**: Class 0 dominates (~86%). SMOTE oversampling will be applied in preprocessing.

### Missing Values
- The raw dataset has no missing values (all features are survey responses coded as integers/floats).

### Outliers
- `BMI`, `MentHlth`, and `PhysHlth` exhibit right-skewed distributions with IQR outliers.
- Outlier capping (Winsorisation) will be applied rather than row removal to preserve data volume.

### Top Predictors
- **HighBP**, **HighChol**, **BMI**, **GenHlth**, and **Age** show the strongest correlations with the target.
- **PhysActivity** and **Fruits/Veggies** are negatively correlated with diabetes.

### Recommendations for Preprocessing
1. Cap outliers using IQR on continuous features (BMI, MentHlth, PhysHlth).
2. Apply StandardScaler — tree models benefit less but LR/NN require it.
3. Use SMOTE to address the class imbalance.
4. Stratified train-test split to preserve class proportions.

Next step: `02_preprocessing.ipynb`